# Load Packages

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Parameters

In [2]:
# Output directory — all plots and tables land here
output_dir = os.path.join("..", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Data directory — all data files land here
data_raw_dir = os.path.join("..", "data", "raw")
data_processed_dir = os.path.join("..", "data", "processed")

# Load Data

In [4]:
# Load data
toaster_sentiment_df = pd.read_csv(os.path.join(data_processed_dir, "toaster_sentiment.csv"))

# Preview data
display(toaster_sentiment_df.head())

toaster_sentiment_df.info()

,ASIN,P_TITLE,OP,DP,SP,FS,PRA_4.5,P_RTG,RTG_P_NO,SELLER_LINK,...,SUBJ,SRVS,CP_RVS,TWRB_SENT,TWRB_SCORE,CHUNKED,SRB_SENT,SRB_SCORE,RVRB_SENT,RVRB_SCORE
0,B01KZ729F6,Hamilton Beach 2 Slice Extra Wide Slot Toaster...,NaN,0.220000,24\n.\n99,1,0,4.4,12579,https://www.amazon.com/stores/HamiltonBeach/pa...,...,0.6,positive,0.6369,positive,0.9670,False,positive,0.9988,positive,0.9982
1,B0744M3SB4,Nostalgia TCS2 Grilled Cheese Toaster with Eas...,209.988477,0.786655,44.8,1,0,4.1,4156,https://www.amazon.com/stores/Nostalgia/page/B...,...,0.675,positive,0.8519,positive,0.8974,False,positive,0.9982,positive,0.9938
2,B0BT5WXBR2,Elite Gourmet ECT118B Cool Touch Single Slice ...,14.990000,0.000000,14.99,1,0,.,.,https://www.amazon.com/stores/EliteGourmet/pag...,...,0.32,positive,0.4404,negative,0.7371,False,negative,0.9995,negative,0.9986
3,B0B9MX21NV,"evoloop Toaster 2 Slice, Stainless Steel Bread...",279.850020,0.874969,34.99,1,0,4.4,31,https://www.amazon.com/stores/evoloop/page/087...,...,0.509831254980509,positive,0.9914,neutral,0.6821,True,positive,0.9978,negative,0.9912
4,B00ZGCKSG8,DASH Clear View Toaster: Extra Wide Slot Toast...,NaN,0.170000,41\n.\n48,1,0,4.4,11283,https://www.amazon.com/stores/DASH/page/F42BA3...,...,.,.,.,NaN,NaN,False,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 62014 entries, 0 to 62013
Data columns (total 37 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ASIN         62014 non-null  str    
 1   P_TITLE      62014 non-null  str    
 2   OP           45643 non-null  float64
 3   DP           62014 non-null  float64
 4   SP           62014 non-null  str    
 5   FS           62014 non-null  str    
 6   PRA_4.5      62014 non-null  int64  
 7   P_RTG        62014 non-null  str    
 8   RTG_P_NO     62014 non-null  str    
 9   SELLER_LINK  62014 non-null  str    
 10  IMAGE_URL    62014 non-null  str    
 11  P_URL        62014 non-null  str    
 12  RV_URL       62014 non-null  str    
 13  PRFL_IMG     62014 non-null  str    
 14  PRFL_URL     62014 non-null  str    
 15  RV_TTL       62011 non-null  str    
 16  RVS          62008 non-null  str    
 17  RVR          62011 non-null  str    
 18  RSR          62014 non-null  int64  
 19  RVR_CONT     62

## Preprocess Data

In [8]:
toaster_sentiment_df["RV_DT"] = pd.to_datetime(toaster_sentiment_df["RV_DT"], errors="coerce")

display(toaster_sentiment_df["RV_DT"].dt.to_period("Q"))

0        2019Q1
1        2018Q4
2        2021Q1
3        2022Q4
4        2020Q4
          ...  
62009    2022Q1
62010    2019Q2
62011    2020Q1
62012    2022Q2
62013    2022Q2
Name: RV_DT, Length: 62014, dtype: period[Q-DEC]

In [ ]:
# Remove missing RVS / RSR values
toaster_sentiment_df = toaster_sentiment_df.dropna(subset=["RVS", "RSR"])

# Remove short reviews (less than 20 characters)
toaster_sentiment_df = toaster_sentiment_df[toaster_sentiment_df["RVS_L"] >= 20].copy()

In [ ]:
# Coerce Review_Date to datetime
toaster_sentiment_df["RS_DT"] = pd.to_datetime(toaster_sentiment_df["Review_Date"], errors="coerce")

# Coerce numeric columns to numeric
num_cols = [
    "OP", "DP", "SP", "FS", "PRA_4.5", "P_RTG", "RT",
    "FS", "HLP_VT", "RVS_L", "SUBJ", "SUBJ_CD",
    "NG_RVS", "NU_RVS", "PS_RVS", "CP_RVS",
    "NGE_RVS", "PSE_RV", "TTL_RV", "RSR", "VP", "IMG_PRST",
]
for col in num_cols:
    toaster_sentiment_df[col] = pd.to_numeric(toaster_sentiment_df[col], errors="coerce")   
    
# Create quarter variables
toaster_sentiment_df["quarter"] = toaster_sentiment_df["RV_DT"].dt.to_period("Q")